# TLIO: Tight Learned Inertial Odometry

- Paper: `arXiv:2007.01867v3` / IEEE Robotics and Automation Letters, 2020
- Authors: Wenxin Liu, David Caruso, Eddy Ilg, Jing Dong, Anastasios I. Mourikis, Kostas Daniilidis, Vijay Kumar, Jakob Engel
- Presentation Author: Javier Penuela 

## Introduction to Pedestrian Dead Reckoning

- Pedestrian dead reckoning (PDR) estimates motion from inertial sensors alone.
- Traditional methods rely on step detection, stride models, and heading estimation.
- Challenges: sensor bias, noise, drift, arbitrary device orientation, and diverse human motion.
- This paper proposes an IMU-only system that avoids step counting by learning short-term displacement priors using a ResNet and physical process modeling.

## Dataset: Collection

- Collected with a custom rig: Bosch BMI055 IMU mounted on a headset rigidly attached to cameras.
- More than 400 sequences, totaling 60 hours of pedestrian data.
- Activities include walking, standing, kitchen tasks, playing pool, stairs, outdoor uneven terrain, and more.
- Data captured with multiple physical devices (visual data for ground true + IMU data) and over 5 people to cover varied motion patterns and IMU biases.
- Limitations: data excludes looking up or down positions, data does not reflect phone conditions, but smart glasses conditions.

## Dataset: Description

- Ground truth from a state-of-the-art visual-inertial filter at 1000 Hz.
- Dataset split randomly into 80% training, 10% validation, 10% test. No difference between validation and test datasets is disclosed
- The dataset contains pedestrian trajectories with 3 to 7 minutes of activity per sequence.
- Benchmark comparison uses 3D-RoNIN as a reference baseline.

## Dataset: Preprocessing

- Training uses overlapping sliding windows of IMU data.
- Each window contains N IMU samples; final choice is `N = 200` for 200 Hz data.
- Data augmentation:
  - random horizontal rotations for yaw invariance (following RoNIN)
  - random sensor bias perturbations
  - random gravity direction perturbations

- Goal: make the network robust to initialization error, bias, and gravity misalignment.

## Dataset: Gravity-Aligned Frame

- IMU samples are rotated to a local gravity-aligned frame built from the orientation at the beginning of each window.
- Gravity-aligned frame ensures gravity points downward and decouples global yaw from local displacement.
- The network input is therefore invariant to arbitrary heading of the headset.

Method:

Compute a reference array `ig_w: np.array([0, 0, 1.0])` representing gravity pointing "up" in the Z-axis of the world frame.
then they compute the rotation matrix $R$ such that $R a = b$:
    - Normalization: It normalizes both the input vector $a$ (the accelerometer reading) and the target vector $b$ (the gravity vector).
    - Rotation Axis ($\omega$): It finds the axis of rotation by taking the cross product of the two vectors: $$\omega = \hat{a} \times \hat{b}$$ This vector $\omega$ is perpendicular to the plane formed by $a$ and $b$.
    - Rodrigues' Rotation Formula: $$R = I + [\omega]\times + \frac{1}{1 + \hat{a} \cdot \hat{b}} ([\omega]_\times)^2$$
    $[\omega]_\times$: The skew-symmetric matrix of $\omega$ (computed by the hat(v) helper function -> `np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])`.
   

## System Design Overview

- Two main components:
  1. Neural network: regresses 3D displacement and uncertainty from IMU segments.
  2. Extended Kalman Filter (EKF): tightly fuses neural displacement measurements with IMU kinematics.

- The network learns a statistical motion prior from data, while the EKF performs model-based propagation.
- IMU data is used twice: directly for state propagation and indirectly as network measurement input. (Data argumentation prevents measurement error propagation to the model)

## Neural Network Description

- Architecture: 1D ResNet18 variant.
- Input: `N x 6` IMU window, where each sample contains accelerometer and gyroscope measurements in gravity-aligned frame.
- Output:
  - $\hat d \in \mathbb{R}^3$: predicted 3D displacement over the window
  - $\hat u \in \mathbb{R}^3$: uncertainty parameters for the displacement covariance

- The network output is treated as a measurement of relative displacement with uncertainty.

## Loss Functions

- Mean Squared Error (MSE) loss:

$$
L_{\text{MSE}} = \frac{1}{n} \sum_{i=1}^n \| d_i - \hat d_i \|^2
$$

- Maximum Likelihood loss for regressed Gaussian displacement:

$$
L_{\text{ML}} = \frac{1}{n} \sum_{i=1}^n \left( \frac{1}{2} \log \det \Sigma_i + \frac{1}{2} (d_i - \hat d_i)^T \Sigma_i^{-1} (d_i - \hat d_i) \right) + \text{const}
$$

- Covariance parametrization by log-standard deviation:

$$
\Sigma_i = \operatorname{diag}\left(e^{2 u_{x,i}}, e^{2 u_{y,i}}, e^{2 u_{z,i}}\right)
$$

- Best performing training strategy: first train with MSE until stable, then switch to likelihood loss.

## Stochastic Cloning

- The EKF state includes the current state and a sliding window of past cloned poses.
- A cloned pose is appended whenever a new network measurement is available.
- This allows the filter to use relative displacement measurements between pairs of past states.

- Full state dimension:

$$
\text{dim}(X) = 6m + 15
$$

where `m` is the number of past cloned states.

- Example: with 20 Hz updates and a 1 s window, up to 21 states are maintained.

## EKF Architecture and Equations

### Propagation model

Strapdown inertial kinematics with IMU bias and gravity:

$$
R_{k+1} = R_k \exp_{\mathrm{SO(3)}}\left((\omega_k - b_{g,k}) \Delta t\right)
$$
$$
v_{k+1} = v_k + g \Delta t + R_k (a_k - b_{a,k}) \Delta t
$$
$$
p_{k+1} = p_k + v_k \Delta t + \frac{1}{2} \left(g + R_k (a_k - b_{a,k}) \right) \Delta t^2
$$
$$
b_{g,k+1} = b_{g,k} + n_{g,k}, \qquad b_{a,k+1} = b_{a,k} + n_{a,k}
$$

### Linearized propagation

$$
\tilde s_{k+1} = A_k \tilde s_k + B_k n_k
$$
$$
P_{k+1} = A_k P_k A_k^T + B_k W B_k^T
$$

### Measurement model in gravity-aligned frame

The network predicts a local displacement measurement:

$$
h(X) = R_i^T (p_j^w - p_i^w) = \hat d_{ij} + d_{ij}
$$

with measurement covariance `\hat \Sigma_{ij}` from the network.

### Kalman update

$$
K = P H^T (H P H^T + \hat \Sigma_{ij})^{-1}
$$
$$
X \leftarrow X \oplus K (h(X) - \hat d_{ij})
$$
$$
P \leftarrow (I - K H) P (I - K H)^T + K \hat \Sigma_{ij} K^T
$$

- Measurement Jacobian `H` has nonzero blocks only for the cloned poses `i` and `j`.
- A `\chi^2` gating test rejects updates with normalized innovation beyond the 99%-tile for 3 DOF.

## Experimental Results

- Evaluation on the held-out test split of the pedestrian dataset.
- Comparison against a state-of-the-art VIO ground truth and the 3D-RoNIN benchmark.
- Results show that combining learned displacement priors with an EKF yields lower drift and better pose estimates than a disconnected concatenation approach.

## RoNIN Benchmark

- RoNIN is the benchmark for robust neural inertial navigation in the wild.
- In this work, 3D-RoNIN refers to a method that takes the same network outputs and concatenates displacements using an AHRS attitude filter.
- 3D-RoNIN is used as the baseline to assess whether the tightly-coupled EKF improves performance.

## Metrics

- Absolute Translation Error (ATE):

$$
\text{ATE} = \sqrt{\frac{1}{n} \sum_{k=1}^n \| p_k^w - \hat p_k^w \|^2}
$$

- Relative Translation Error (RTE) over window `t`.
- Drift rate `DR(%)` over trajectory length.
- Absolute Yaw Error (AYE).
- Relative Yaw Error (RYE) per unit time.

- The paper reports cumulative distribution functions of these metrics over the test set.

## System Performance

- TLIO consistently outperforms 3D-RoNIN on all reported metrics.
- Higher filter update frequency improves ATE and drift, despite measurement correlation.
- The EKF also reduces yaw drift compared to a decoupled attitude filter.
- Key advantages:
  - learned measurement uncertainty improves fusion
  - filter robustness to outliers via `\chi^2` gating
  - no handcrafted step detection or gait model required

## Presenter Comments

- This paper shows a strong hybrid approach: learned priors plus classical filtering.
- The EKF uses both model-based IMU propagation and data-driven displacement measurements.
- Gravity-aligned measurements protect the filter from false yaw observability.
- The system is trained on pedestrian motion and remains limited by the training distribution.
- Unusual motions not present in training can still cause failure cases.
- Overall, TLIO demonstrates that IMU-only pose estimation can be improved by learning measurement uncertainty and integrating it tightly in an EKF.